# minatar_deterministic

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load each game's stored DQN runs and the two published DQN references.
COMPONENT = "dqn_minatar"
component = EXPERIMENT.component(COMPONENT)
GAMES = component.sweep["ENV_HYPERS.GAME"]
TOTAL_STEPS = component.config.AGENT_HYPERS.TOTAL_TIMESTEPS

results = {
    game: load_results(EXPERIMENT, COMPONENT, where={"ENV_HYPERS.GAME": game})
    for game in GAMES
}
results = {game: runs for game, runs in results.items() if runs is not None}
reg_duel_q = json.loads(Path("reg_duel_q_dqn_reference.json").read_text())
paper = json.loads(Path("minatar_paper_dqn_reference.json").read_text())

In [ ]:
# Compute each game's mean return over steps with a 95% bootstrap CI, and the
# references up to our last step. reg-duel-q's is evaluation score with its mean
# and 95% band, the paper's is the training return of its figure.
curves = {}
for game, runs in results.items():
    steps, returns = subsample(return_curve(runs.reward, runs.done))
    curves[game] = (steps, mean_ci(returns))

reg_duel_q_curves = {}
paper_curves = {}
for game in GAMES:
    ref_steps = np.asarray(reg_duel_q[game]["steps"])
    mean = np.asarray(reg_duel_q[game]["mean_return"])
    band = 1.96 * np.asarray(reg_duel_q[game]["stderr"])
    in_window = ref_steps <= TOTAL_STEPS
    reg_duel_q_curves[game] = (
        ref_steps[in_window],
        mean[in_window],
        (mean - band)[in_window],
        (mean + band)[in_window],
    )
    paper_steps = np.asarray(paper[game]["steps"])
    in_window = paper_steps <= TOTAL_STEPS
    paper_curves[game] = (
        paper_steps[in_window],
        np.asarray(paper[game]["mean_return"])[in_window],
    )

In [ ]:
# Style for the learning-curve plots. Every piece of text shares one size.
FONT_SIZE = 32
STYLE = {
    "figure.figsize": (32, 8),
    "font.family": "Times New Roman",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
}
BAND_ALPHA = 0.2
DQN_COLOR = "tab:blue"
REG_DUEL_Q_COLOR = "tab:gray"
PAPER_COLOR = "tab:olive"
# Only the left panel carries the legend, since the panels share colors.
LEGEND = {
    "loc": "lower right",
    "bbox_to_anchor": (1.04, -0.04),
    "handlelength": 1,
    "handletextpad": 0.4,
}
LEGEND_MARKER = {"marker": "o", "linestyle": "none", "markersize": 14}
BOX_ASPECT = 1
# Ticks sit only at the axis ends, so the labels can tuck in between them.
LABEL_PAD = -20
XLABEL = "Time Steps"
XTICKS = [0, TOTAL_STEPS]
XTICK_LABELS = ["0", f"{TOTAL_STEPS // 1_000_000}M"]
YLABEL = "Return"
YTICK_STEP = 10


def title(game):
    return game.replace("_", " ").title()


def top_of_axis(game):
    # The axis ends at the next multiple of YTICK_STEP above every curve drawn.
    highest = max(reg_duel_q_curves[game][3].max(), paper_curves[game][1].max())
    if game in curves:
        highest = max(highest, np.nanmax(curves[game][1].high))
    return int(np.ceil(highest / YTICK_STEP) * YTICK_STEP)

In [ ]:
# Plot every game's panel side by side. Each panel has its own y-axis, so each
# is labelled. Our curve is drawn over the references where runs are stored.
with plt.rc_context(STYLE):
    fig, axes = plt.subplots(1, len(GAMES))
    for ax, game in zip(axes, GAMES, strict=True):
        ax.set_box_aspect(BOX_ASPECT)
        ref_steps, ref_mean, ref_low, ref_high = reg_duel_q_curves[game]
        ax.fill_between(
            ref_steps, ref_low, ref_high, color=REG_DUEL_Q_COLOR, alpha=BAND_ALPHA
        )
        ax.plot(ref_steps, ref_mean, color=REG_DUEL_Q_COLOR)
        ax.plot(*paper_curves[game], color=PAPER_COLOR)
        if game in curves:
            steps, (mean, low, high) = curves[game]
            ax.fill_between(steps, low, high, color=DQN_COLOR, alpha=BAND_ALPHA)
            ax.plot(steps, mean, color=DQN_COLOR)
        top = top_of_axis(game)
        ax.set_title(title(game))
        ax.set_xlabel(XLABEL, labelpad=LABEL_PAD)
        ax.set_xticks(XTICKS, XTICK_LABELS)
        ax.set_yticks([0, top], ["0", str(top)])
        ax.set_ylim(0, top)
        ax.set_ylabel(YLABEL, labelpad=LABEL_PAD)
    fig.tight_layout()
    entries = [
        ("reg-duel-q ref", REG_DUEL_Q_COLOR),
        ("minatar-paper ref\n(extrapolated)", PAPER_COLOR),
    ]
    if curves:
        entries.insert(0, ("DQN", DQN_COLOR))
    handles = [Line2D([], [], color=color, **LEGEND_MARKER) for _, color in entries]
    axes[0].legend(handles, [label for label, _ in entries], **LEGEND)
figures = {"minatar_dqn_curves": fig}
plt.show()

In [ ]:
# Save every plotted figure as PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    for ext in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=200)
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")